# Notebook 4.2 - ConvMixer Baseline

Notebook này chạy riêng model `convmixer_256_8` với 5 seed `42-46` để lấy số liệu bảng so sánh.


In [1]:
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent

sys.path.append(str(repo_root))

from src.baseline_experiment import run_baseline_suite
from src.baseline_protocol import (
    DEFAULT_BASELINE_SEEDS,
    build_output_dir,
    ensure_seed_completeness,
    filter_runs_for_run,
    make_fair_train_config,
)

data_dir = repo_root / 'data' / 'features' / 'mel'
base_output_dir = repo_root / 'data' / 'models' / 'baselines'

RUN_ID = 'paper_v1'
output_dir = build_output_dir(base_output_dir=base_output_dir, run_id=RUN_ID)

print(f'Repo root: {repo_root}')
print(f'Data dir: {data_dir}')
print(f'Output dir: {output_dir}')
print(f'Run ID: {RUN_ID}')


Repo root: /home/anhcbt/extend/workspace/convmixer_model
Data dir: /home/anhcbt/extend/workspace/convmixer_model/data/features/mel
Output dir: /home/anhcbt/extend/workspace/convmixer_model/data/models/baselines/paper_v1
Run ID: paper_v1


/home/anhcbt/extend/workspace/convmixer_model/src/models/ast_official_models.py:204: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast()
/home/anhcbt/extend/workspace/convmixer_model/.venv/lib/python3.13/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


## Protocol

- Model: `convmixer_256_8`
- Seeds: `42, 43, 44, 45, 46`
- Mục tiêu: lấy số liệu cho bảng, không vẽ biểu đồ.


In [2]:
selected_model = 'convmixer_256_8'
model_label = 'ConvMixer-256/8'
selected_seeds = DEFAULT_BASELINE_SEEDS
USE_CACHED_RESULTS = False  # True: chỉ đọc CSV trong đúng run_id, không train

config = make_fair_train_config(
    data_dir=data_dir,
    base_output_dir=base_output_dir,
    run_id=RUN_ID,
    model_names=(selected_model,),
    seeds=selected_seeds,
    include_existing_runs=True,
    skip_completed_runs=True,
)

config


TrainConfig(data_dir='/home/anhcbt/extend/workspace/convmixer_model/data/features/mel', output_dir='/home/anhcbt/extend/workspace/convmixer_model/data/models/baselines/paper_v1', run_id='paper_v1', data_version='paper_v1', model_names=('convmixer_256_8',), seeds=(42, 43, 44, 45, 46), split_seed=42, train_ratio=0.75, val_ratio=0.15, batch_size=32, num_workers=4, num_epochs=25, lr=0.001, weight_decay=0.0001, patience=5, min_delta=0.02, scheduler_factor=0.2, scheduler_patience=3, scheduler_min_lr=1e-06, ast_official_model_size='tiny224', ast_official_fstride=10, ast_official_tstride=10, ast_official_input_fdim=128, ast_official_input_tdim=32, ast_official_imagenet_pretrain=False, ast_official_audioset_pretrain=False, ast_official_verbose=True, ast_official_auto_input_shape=True, ast_official_auto_norm_from_train=True, ast_official_norm_mean=None, ast_official_norm_std=None, ast_official_lr=0.001, ast_official_weight_decay=0.0001, use_model_specific_hparams=False, include_existing_runs=Tru

In [3]:
def build_single_summary(runs: pd.DataFrame) -> pd.DataFrame:
    if runs.empty:
        return pd.DataFrame()
    return (
        runs.groupby('model', as_index=False)
        .agg(
            n_runs=('seed', 'nunique'),
            params=('params', 'mean'),
            test_accuracy_mean=('test_accuracy', 'mean'),
            test_accuracy_std=('test_accuracy', 'std'),
            macro_f1_mean=('macro_f1', 'mean'),
            macro_f1_std=('macro_f1', 'std'),
            weighted_f1_mean=('weighted_f1', 'mean'),
            weighted_f1_std=('weighted_f1', 'std'),
            train_seconds_mean=('train_seconds', 'mean'),
            infer_seconds_mean=('infer_seconds', 'mean'),
        )
        .fillna(0.0)
    )

if USE_CACHED_RESULTS:
    print('[MODE] Cached: đọc CSV hiện có trong run_id, không train lại.')
    runs_df = pd.read_csv(output_dir / 'baseline_runs.csv')
    history_store = {}
    metadata = {
        'mode': 'cached',
        'run_id': RUN_ID,
        'runs_csv': str(output_dir / 'baseline_runs.csv'),
    }
else:
    print('[MODE] Train: train phần thiếu trong run_id hiện tại.')
    runs_df, _, history_store, metadata = run_baseline_suite(config)

runs_df = filter_runs_for_run(
    runs_df=runs_df,
    model_names=(selected_model,),
    run_id=RUN_ID,
)
ensure_seed_completeness(
    runs_df=runs_df,
    model_names=(selected_model,),
    seeds=selected_seeds,
)
summary_df = build_single_summary(runs_df)

runs_selected_path = output_dir / f'baseline_runs_{selected_model}.csv'
summary_selected_path = output_dir / f'baseline_summary_{selected_model}.csv'

runs_df.to_csv(runs_selected_path, index=False)
summary_df.to_csv(summary_selected_path, index=False)

print('Metadata:')
print(metadata)
print('Saved:', runs_selected_path)
print('Saved:', summary_selected_path)


[MODE] Train: train phần thiếu trong run_id hiện tại.
[START] model=convmixer_256_8 seed=42 params=719117 device=cpu lr=0.001 wd=0.0001
[convmixer_256_8|seed42] Epoch 1/25 | train_loss=1.7774 | val_loss=1.2193 | val_acc=48.42% | lr=0.001000->0.001000 | improved
[convmixer_256_8|seed42] Epoch 2/25 | train_loss=1.0379 | val_loss=0.7207 | val_acc=71.73% | lr=0.001000->0.001000 | improved
[convmixer_256_8|seed42] Epoch 3/25 | train_loss=0.6817 | val_loss=0.4166 | val_acc=86.02% | lr=0.001000->0.001000 | improved
[convmixer_256_8|seed42] Epoch 4/25 | train_loss=0.5196 | val_loss=0.3964 | val_acc=85.26% | lr=0.001000->0.001000 | no_improve(1/5)
[convmixer_256_8|seed42] Epoch 5/25 | train_loss=0.4059 | val_loss=0.2857 | val_acc=90.38% | lr=0.001000->0.001000 | improved
[convmixer_256_8|seed42] Epoch 6/25 | train_loss=0.3399 | val_loss=0.3172 | val_acc=90.23% | lr=0.001000->0.001000 | no_improve(1/5)
[convmixer_256_8|seed42] Epoch 7/25 | train_loss=0.2923 | val_loss=0.2427 | val_acc=92.03% | l

In [4]:
display_cols = [
    'model', 'seed', 'params', 'best_epoch', 'test_accuracy',
    'macro_f1', 'weighted_f1', 'infer_seconds', 'train_seconds'
]

runs_df[display_cols].sort_values('seed')


,model,seed,params,best_epoch,test_accuracy,macro_f1,weighted_f1,infer_seconds,train_seconds
0,convmixer_256_8,42,719117,18,96.475771,0.964692,0.964658,0.375855,369.769848
1,convmixer_256_8,43,719117,25,96.696035,0.966840,0.966994,0.333949,396.210293
2,convmixer_256_8,44,719117,11,96.696035,0.966949,0.966901,0.435410,246.666843
3,convmixer_256_8,45,719117,22,96.696035,0.967077,0.967082,0.326976,404.232487
4,convmixer_256_8,46,719117,24,96.916300,0.969060,0.969157,0.352358,399.798624


In [5]:
params_m = runs_df['params'].mean() / 1_000_000

acc_mean = runs_df['test_accuracy'].mean()
acc_std = runs_df['test_accuracy'].std(ddof=1)
macro_f1_mean = runs_df['macro_f1'].mean()
macro_f1_std = runs_df['macro_f1'].std(ddof=1)
weighted_f1_mean = runs_df['weighted_f1'].mean()
weighted_f1_std = runs_df['weighted_f1'].std(ddof=1)
infer_mean = runs_df['infer_seconds'].mean()
infer_std = runs_df['infer_seconds'].std(ddof=1)
train_mean = runs_df['train_seconds'].mean()
train_std = runs_df['train_seconds'].std(ddof=1)

latex_row = (
    f"{model_label} & {params_m:.2f} & "
    f"{acc_mean:.2f}$\\pm${acc_std:.2f} & "
    f"{macro_f1_mean:.4f}$\\pm${macro_f1_std:.4f} & "
    f"{weighted_f1_mean:.4f}$\\pm${weighted_f1_std:.4f} & "
    f"{infer_mean:.4f}$\\pm${infer_std:.4f} & "
    f"{train_mean:.1f}$\\pm${train_std:.1f} \\\\"
)

result_df = pd.DataFrame([
    {
        'model_label': model_label,
        'params_m': params_m,
        'test_accuracy_mean': acc_mean,
        'test_accuracy_std': acc_std,
        'macro_f1_mean': macro_f1_mean,
        'macro_f1_std': macro_f1_std,
        'weighted_f1_mean': weighted_f1_mean,
        'weighted_f1_std': weighted_f1_std,
        'infer_seconds_mean': infer_mean,
        'infer_seconds_std': infer_std,
        'train_seconds_mean': train_mean,
        'train_seconds_std': train_std,
    }
])

metrics_path = output_dir / f'baseline_table_metrics_{selected_model}.csv'
result_df.to_csv(metrics_path, index=False)

print(latex_row)
print('Saved:', metrics_path)
result_df


ConvMixer-256/8 & 0.72 & 96.70$\pm$0.16 & 0.9669$\pm$0.0015 & 0.9670$\pm$0.0016 & 0.3649$\pm$0.0437 & 363.3$\pm$66.6 \\
Saved: /home/anhcbt/extend/workspace/convmixer_model/data/models/baselines/paper_v1/baseline_table_metrics_convmixer_256_8.csv


,model_label,params_m,test_accuracy_mean,test_accuracy_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,infer_seconds_mean,infer_seconds_std,train_seconds_mean,train_seconds_std
0,ConvMixer-256/8,0.719117,96.696035,0.15575,0.966924,0.001547,0.966958,0.001593,0.36491,0.043718,363.335619,66.588008
